In [1]:
# # Step 1: Remove corrupted numpy completely
# !rm -rf /opt/conda/lib/python3.10/site-packages/numpy*
# !rm -rf /opt/conda/lib/python3.10/site-packages/numpy-*

# # Step 2: Reinstall the working versions
# !pip install numpy==1.23.5 matplotlib==3.6.3 scikit-learn==1.1.3 imbalanced-learn==0.10.1




In [ ]:
import os
import time
import pickle

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fftpack import dct

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE

In [ ]:


# Replace 'your_file.csv' with the actual path to your CSV file
file_path = r'/kaggle/input/auitd-dataset/Labels_AUITD.csv'

column_names = ['Image', 'Stage_1', 'Stage_2']

df = pd.read_csv(file_path, header=None, names=column_names)

df=df[1:792]

print(df)

     Image Stage_1 Stage_2
1    2_100       1       2
2    2_101       1       2
3    2_102       1       2
4    2_103       1       2
5    2_104       1       2
..     ...     ...     ...
787   5_95       2       5
788   5_96       2       5
789   5_97       2       5
790   5_98       2       5
791   5_99       2       5

[791 rows x 3 columns]


In [ ]:

rootdir = r'/kaggle/input/auitd-dataset/All_images/All_images'
image_path = []
files = os.listdir(rootdir)
for file in files:
    image_path.append(rootdir + '/' + file)

dct_df = pd.DataFrame(columns=['features', 'labels_1','labels_2' ])


def zigzag_indices(n):
    index_order = sorted(((x, y) for x in range(n) for y in range(n)),
                         key=lambda s: (s[0] + s[1], s[1] if (s[0] + s[1]) % 2 else -s[1]))
    return [(x, y) for x, y in index_order]

def extract_dct_descriptors(image):
    # Resize and convert to grayscale
    resized_image = cv2.resize(image, (256, 256))
    gray_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)
    
    # Apply DCT on the entire image
    dct_image = cv2.dct(np.float32(gray_image))
    
    # Get zigzag order for the entire 256x256 image
    zigzag_order = zigzag_indices(256)
    
    # Flatten the DCT coefficients according to the zigzag order
    flattened_dct = [dct_image[idx] for idx in zigzag_order]
    
    # Keep only the first 33405 coefficients (low and mid-frequency)
    truncated_dct = flattened_dct[:32896]
    
    # Convert to numpy array and return
    descriptors_array = np.array(truncated_dct)
    return descriptors_array

# Define a global dataframe
dct_df = pd.DataFrame(columns=['image_name', 'features', 'labels_1', 'labels_2'])

def main():
    global dct_df
    t0 = time.time()

    for file in image_path:
        img = cv2.imread(file)

        # Extract image name
        img_name = os.path.basename(file).split('.')[0]

        # Extract labels from the DataFrame (df)
        label_1 = df.loc[df['Image'] == img_name, 'Stage_1'].iloc[0]
        label_2 = df.loc[df['Image'] == img_name, 'Stage_2'].iloc[0]

        # Extract features
        img_des = extract_dct_descriptors(img)

        if img_des is not None:
            new_data = pd.DataFrame({
                'image_name': [img_name],
                'features': [img_des],
                'labels_1': [label_1],
                'labels_2': [label_2]
            })

            dct_df = pd.concat([dct_df, new_data], ignore_index=True)

    # print('Time taken: {:.2f} seconds'.format(time.time() - t0))

print('\nCalculating')
main()

print(dct_df)


Calculating
    image_name                                           features labels_1  \
0         5_59  [31793.812, 854.94867, -124.86836, -438.14948,...        2   
1        4_316  [31895.844, 1562.6635, -925.14197, -877.0686, ...        2   
2        4_122  [31174.36, 2318.5647, -525.1632, 249.86043, 96...        2   
3        4_388  [25891.672, 7103.078, -22.260078, -2637.0918, ...        2   
4         4_77  [27561.406, 5610.4673, 1729.494, -56.783783, -...        2   
..         ...                                                ...      ...   
776       3_87  [28487.172, 5344.1826, 1797.9167, -1734.977, -...        1   
777       2_87  [30400.355, 2337.9905, 280.221, -1000.4753, -2...        1   
778       3_62  [28286.09, 5192.178, -1374.0469, -328.37042, 1...        1   
779       5_80  [29229.79, 4226.53, -1189.9359, 645.5901, 1416...        2   
780       2_77  [31681.918, 1605.9446, -91.02339, -290.71765, ...        1   

    labels_2  
0          5  
1          4  
2    

In [5]:
print("Length of features in hog df", len(dct_df['features'].iloc[-1]))

Length of features in hog df 32896


In [6]:
dct_df

,image_name,features,labels_1,labels_2
0,5_59,"[31793.812, 854.94867, -124.86836, -438.14948,...",2,5
1,4_316,"[31895.844, 1562.6635, -925.14197, -877.0686, ...",2,4
2,4_122,"[31174.36, 2318.5647, -525.1632, 249.86043, 96...",2,4
3,4_388,"[25891.672, 7103.078, -22.260078, -2637.0918, ...",2,4
4,4_77,"[27561.406, 5610.4673, 1729.494, -56.783783, -...",2,4
...,...,...,...,...
776,3_87,"[28487.172, 5344.1826, 1797.9167, -1734.977, -...",1,3
777,2_87,"[30400.355, 2337.9905, 280.221, -1000.4753, -2...",1,2
778,3_62,"[28286.09, 5192.178, -1374.0469, -328.37042, 1...",1,3
779,5_80,"[29229.79, 4226.53, -1189.9359, 645.5901, 1416...",2,5


In [7]:

# Dropping the 'labels_1' column
dct_df_stage_1 = dct_df.drop(columns=['labels_2'])
# Dropping the 'labels_1' column
dct_df_stage_2 = dct_df.drop(columns=['labels_1'])



In [8]:
# Display the updated DataFrame
print(dct_df_stage_1)

    image_name                                           features labels_1
0         5_59  [31793.812, 854.94867, -124.86836, -438.14948,...        2
1        4_316  [31895.844, 1562.6635, -925.14197, -877.0686, ...        2
2        4_122  [31174.36, 2318.5647, -525.1632, 249.86043, 96...        2
3        4_388  [25891.672, 7103.078, -22.260078, -2637.0918, ...        2
4         4_77  [27561.406, 5610.4673, 1729.494, -56.783783, -...        2
..         ...                                                ...      ...
776       3_87  [28487.172, 5344.1826, 1797.9167, -1734.977, -...        1
777       2_87  [30400.355, 2337.9905, 280.221, -1000.4753, -2...        1
778       3_62  [28286.09, 5192.178, -1374.0469, -328.37042, 1...        1
779       5_80  [29229.79, 4226.53, -1189.9359, 645.5901, 1416...        2
780       2_77  [31681.918, 1605.9446, -91.02339, -290.71765, ...        1

[781 rows x 3 columns]


In [9]:
# Display the updated DataFrame
print(dct_df_stage_2)

    image_name                                           features labels_2
0         5_59  [31793.812, 854.94867, -124.86836, -438.14948,...        5
1        4_316  [31895.844, 1562.6635, -925.14197, -877.0686, ...        4
2        4_122  [31174.36, 2318.5647, -525.1632, 249.86043, 96...        4
3        4_388  [25891.672, 7103.078, -22.260078, -2637.0918, ...        4
4         4_77  [27561.406, 5610.4673, 1729.494, -56.783783, -...        4
..         ...                                                ...      ...
776       3_87  [28487.172, 5344.1826, 1797.9167, -1734.977, -...        3
777       2_87  [30400.355, 2337.9905, 280.221, -1000.4753, -2...        2
778       3_62  [28286.09, 5192.178, -1374.0469, -328.37042, 1...        3
779       5_80  [29229.79, 4226.53, -1189.9359, 645.5901, 1416...        5
780       2_77  [31681.918, 1605.9446, -91.02339, -290.71765, ...        2

[781 rows x 3 columns]


In [ ]:


# Assuming dct_df_stage_2 is already defined and contains the columns 'features' and 'labels_2'

# Convert labels_2 to numeric if it's not already
dct_df_stage_2['labels_2'] = pd.to_numeric(dct_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 2 or 3
dct_df_benign = dct_df_stage_2[dct_df_stage_2['labels_2'].isin([2, 3])]

# Display the filtered dataframe
print(dct_df_benign)


    image_name                                           features  labels_2
5         3_95  [29414.496, 3559.4895, -1764.7393, -1204.936, ...         3
9         2_57  [27699.387, 4171.7837, 1278.4243, -1372.8362, ...         2
14        3_77  [30295.531, 3619.202, -1074.6111, -149.17181, ...         3
16        3_42  [32523.062, 596.1724, 430.7782, -585.6184, -17...         3
17        3_43  [28894.36, 3313.8835, -2076.0828, -4140.6523, ...         3
..         ...                                                ...       ...
775      2_110  [30900.004, 2582.7947, 435.29623, -919.69324, ...         2
776       3_87  [28487.172, 5344.1826, 1797.9167, -1734.977, -...         3
777       2_87  [30400.355, 2337.9905, 280.221, -1000.4753, -2...         2
778       3_62  [28286.09, 5192.178, -1374.0469, -328.37042, 1...         3
780       2_77  [31681.918, 1605.9446, -91.02339, -290.71765, ...         2

[182 rows x 3 columns]


In [ ]:


# Assuming dct_df_stage_2 is already defined and contains the columns 'features' and 'labels_2'

# Convert labels_2 to numeric if it's not already
dct_df_stage_2['labels_2'] = pd.to_numeric(dct_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 4 or 5
dct_df_malignant = dct_df_stage_2[dct_df_stage_2['labels_2'].isin([4,5])]

# Display the filtered dataframe
print(dct_df_malignant)


    image_name                                           features  labels_2
0         5_59  [31793.812, 854.94867, -124.86836, -438.14948,...         5
1        4_316  [31895.844, 1562.6635, -925.14197, -877.0686, ...         4
2        4_122  [31174.36, 2318.5647, -525.1632, 249.86043, 96...         4
3        4_388  [25891.672, 7103.078, -22.260078, -2637.0918, ...         4
4         4_77  [27561.406, 5610.4673, 1729.494, -56.783783, -...         4
..         ...                                                ...       ...
767       4_13  [31398.598, 951.1391, 654.39685, 28.497562, -3...         4
769      4_493  [29933.195, 3926.82, -612.1252, -1665.1294, 10...         4
771      4_156  [29481.398, 4129.4204, -267.03165, 1882.4204, ...         4
772      4_248  [27224.434, 5703.9766, -3119.1812, -1272.9976,...         4
779       5_80  [29229.79, 4226.53, -1189.9359, 645.5901, 1416...         5

[599 rows x 3 columns]


# Stage 1 Benign/Malignant


In [ ]:
##only train oversampled 



# =====================================================
# Ensure labels are int
# =====================================================

dct_df_stage_1['labels_1'] = (
    dct_df_stage_1['labels_1'].astype(int)
)

# =====================================================
# Separate features, labels, names
# =====================================================

X = np.array(
    dct_df_stage_1['features'].tolist()
)

y = np.array(
    dct_df_stage_1['labels_1']
)

names = np.array(
    dct_df_stage_1['image_name']
)

# =====================================================
# Kaggle output root
# =====================================================

output_root = "/kaggle/working"

model_dir = os.path.join(
    output_root,
    "models_stage_1"
)

result_dir = os.path.join(
    output_root,
    "results_stage_1"
)

os.makedirs(model_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

# =====================================================
# Parameters
# =====================================================

num_splits = 5
kernel = 'rbf'

# =====================================================
# Metric storage
# =====================================================

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

# =====================================================
# Stratified KFold
# =====================================================

kfold = StratifiedKFold(
    n_splits=num_splits,
    shuffle=True
)

# =====================================================
# Store predictions
# =====================================================

all_results = pd.DataFrame(
    columns=[
        'fold',
        'image_name',
        'true_label',
        'predicted_label'
    ]
)

# =====================================================
# Cross-validation
# =====================================================

for fold, (train_index, test_index) in enumerate(
        kfold.split(X, y), start=1):

    print("\n" + "="*50)
    print(f"Fold {fold}")
    print("="*50)

    # =================================================
    # Split data
    # =================================================

    X_train = X[train_index]
    X_test  = X[test_index]

    y_train = y[train_index]
    y_test  = y[test_index]

    names_test = names[test_index]

    # =================================================
    # Original Test Distribution
    # =================================================

    unique_test, counts_test = np.unique(
        y_test,
        return_counts=True
    )

    print("\nOriginal Test Distribution:")

    for cls, cnt in zip(unique_test, counts_test):
        print(f"Class {cls}: {cnt}")

    print(f"\nTotal Test Samples: {len(y_test)}")

    # =================================================
    # SMOTE on TRAIN
    # =================================================

    smote_train = SMOTE(random_state=5)

    X_train_smote, y_train_smote = (
        smote_train.fit_resample(
            X_train,
            y_train
        )
    )

    # =================================================
    # Training Distribution After SMOTE
    # =================================================

    print("\nTraining Distribution After SMOTE:")

    unique_sm, counts_sm = np.unique(
        y_train_smote,
        return_counts=True
    )

    for cls, cnt in zip(unique_sm, counts_sm):
        print(f"Class {cls}: {cnt}")

    X_test_final = X_test
    y_test_final = y_test

    # =================================================
    # TRAIN SVM
    # =================================================

    svm = SVC(
        kernel=kernel
    )

    svm.fit(
        X_train_smote,
        y_train_smote
    )

    # =================================================
    # PREDICT
    # =================================================

    y_pred = svm.predict(
        X_test_final
    )

    # =================================================
    # Save model
    # =================================================

    model_path = os.path.join(
        model_dir,
        f"svm_fold_{fold}.pkl"
    )

    with open(model_path, 'wb') as f:
        pickle.dump(svm, f)

    # =================================================
    # Save fold predictions
    # =================================================

    fold_df = pd.DataFrame({
        'fold': [fold] * len(y_test_final),
        'image_name': ['mixed_test']
                      * len(y_test_final),
        'true_label': y_test_final,
        'predicted_label': y_pred
    })

    all_results = pd.concat(
        [all_results, fold_df],
        ignore_index=True
    )

    # =================================================
    # Confusion Matrix
    # =================================================

    cm = confusion_matrix(
        y_test_final,
        y_pred,
    )

    tn, fp, fn, tp = cm.ravel()

    # =================================================
    # Metrics
    # =================================================

    specificity = (
        tn / (tn + fp + 1e-10)
    )

    sensitivity = (
        tp / (tp + fn + 1e-10)
    )

    accuracy = (
        (tp + tn)
        / (tp + tn + fp + fn + 1e-10)
    )

    precision = (
        tp / (tp + fp + 1e-10)
    )

    f1_score = (
        2 * (precision * sensitivity)
    ) / (
        precision + sensitivity + 1e-10
    )

    # =================================================
    # Store metrics
    # =================================================

    specificity_scores.append(
        specificity
    )

    sensitivity_scores.append(
        sensitivity
    )

    accuracy_scores.append(
        accuracy
    )

    precision_scores.append(
        precision
    )

    f1_scores.append(
        f1_score
    )

# =====================================================
# SAVE CSV
# =====================================================

csv_path = os.path.join(
    result_dir,
    "fold_predictions.csv"
)

all_results.to_csv(
    csv_path,
    index=False
)

# =====================================================
# Average Metrics
# =====================================================

average_accuracy = np.mean(
    accuracy_scores
)

average_specificity = np.mean(
    specificity_scores
)

average_sensitivity = np.mean(
    sensitivity_scores
)

average_precision = np.mean(
    precision_scores
)

average_f1_score = np.mean(
    f1_scores
)

# =====================================================
# FINAL RESULTS
# =====================================================

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print("\nAverage Precision:",
      average_precision)

print("Average F1 Score:",
      average_f1_score)

print("\nAverage Specificity:",
      average_specificity)

print("Average Sensitivity (Recall):",
      average_sensitivity)

print("Average Accuracy:",
      average_accuracy)


Fold 1

Original Test Distribution:
Class 1: 37
Class 2: 120

Total Test Samples: 157

Training Distribution After SMOTE:
Class 1: 479
Class 2: 479

Fold 2

Original Test Distribution:
Class 1: 36
Class 2: 120

Total Test Samples: 156

Training Distribution After SMOTE:
Class 1: 479
Class 2: 479

Fold 3

Original Test Distribution:
Class 1: 36
Class 2: 120

Total Test Samples: 156

Training Distribution After SMOTE:
Class 1: 479
Class 2: 479

Fold 4

Original Test Distribution:
Class 1: 36
Class 2: 120

Total Test Samples: 156

Training Distribution After SMOTE:
Class 1: 479
Class 2: 479

Fold 5

Original Test Distribution:
Class 1: 37
Class 2: 119

Total Test Samples: 156

Training Distribution After SMOTE:
Class 1: 480
Class 2: 480

FINAL RESULTS

Average Precision: 0.7367261618803358
Average F1 Score: 0.8410456829300014

Average Specificity: 0.6494677871143038
Average Sensitivity (Recall): 0.9799719887947003
Average Accuracy: 0.8147198879548421
